# Portfolio Biodiversity Impact Score (PBIS)

Code for the paper *Portfolio Biodiversity Impact Assessment and Unsupervised Learning for Banks in Indonesia, Malaysia, and Singapore*.

The notebook reads three files from the `data` folder:

- `credit_sector.xlsx`: gross loans by reported sector for 13 banks, 2023-2025, with the ISIC sections each category is mapped to
- `encore_raw.xlsx`: ENCORE pressure ratings (October 2024 release)
- `env_lseg.xlsx`: LSEG Environmental Pillar Scores. This file is licensed and not included; without it, Section 3 and Fig. 3 are skipped.

Tables go to `outputs/tables`, figures to `outputs/figures`, and the numbers used in the paper to `outputs/paper_numbers.json`. A full run takes a few minutes because of the bootstrap and permutation loops; lower `N_BOOT` and `N_PERM` for a quick check.

In [ ]:
import os, glob, json, re, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import brentq
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option("display.width", 200, "display.max_columns", 30)

# ================================ SETTINGS ================================
SEED   = 42
N_BOOT = 2000            # bootstrap replications (resampling banks)
N_PERM = 10000           # permutation replications
K_RANGE = [2, 3, 4, 5]   # cluster numbers examined
PCA_VAR = 0.80           # share of variance retained before clustering
RATING  = {"VL": 1, "L": 2, "M": 3, "H": 4, "VH": 5}

# Baseline mapping assumptions (each one is varied in Section 2.5)
ID_OTHERS_AS_HOUSEHOLD = True   # Indonesian banks' "Others" holds household/consumer loans
SG_BC_REAL_ESTATE      = 0.5    # share of Singapore "Building and construction" treated as real estate (L)

ENCORE_VERSION = "ENCORE knowledge base, October 2024 release (09. Pressure materiality detail)"
DATA_DIR = "data"            # folder with the input files

OUT = "/content/outputs" if os.path.isdir("/content") else "./outputs"
FIG, TAB = f"{OUT}/figures", f"{OUT}/tables"
for d in (OUT, FIG, TAB): os.makedirs(d, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "font.size": 8,
                     "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 6.5, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "savefig.dpi": 300, "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": .25})
COLORS = {"Indonesia": "#d62728", "Malaysia": "#1f77b4", "Singapore": "#2ca02c"}
NUM = {}   # numbers quoted in the paper; saved to outputs/paper_numbers.json
rng = np.random.default_rng(SEED)   # one generator for all bootstrap and permutation loops

def save_fig(name):
    plt.savefig(f"{FIG}/{name}.png"); plt.savefig(f"{FIG}/{name}.pdf"); plt.show() if "inline" in matplotlib.get_backend() else None; plt.close()
def save_tab(df, name, index=True):
    df.to_csv(f"{TAB}/{name}.csv", index=index); return df

In [ ]:
FILES = {"credit": "credit_sector.xlsx", "encore": "encore_raw.xlsx", "env": "env_lseg.xlsx"}

def find_inputs():
    roots = [DATA_DIR, ".", "/content", f"/content/{DATA_DIR}", "/content/drive/MyDrive"]
    found = {}
    for key, name in FILES.items():
        for r in roots:
            if os.path.isfile(os.path.join(r, name)):
                found[key] = os.path.join(r, name)
                break
    return found

PATH = find_inputs()
if not {"credit", "encore"} <= set(PATH):
    try:   # on Colab, ask for the files
        from google.colab import files
        print("Upload credit_sector.xlsx and encore_raw.xlsx (and env_lseg.xlsx if you have it)")
        files.upload(); PATH = find_inputs()
    except ImportError:
        pass
assert {"credit", "encore"} <= set(PATH), f"Missing input file(s): {set(FILES) - set(PATH) - {'env'}}"

# The LSEG scores are licensed and not part of the repository. Without them the notebook
# still runs everything except RQ2 (Section 3) and Fig. 3.
HAS_ENV = "env" in PATH
for k, v in PATH.items(): print(f"{k:7s} -> {v}")
if not HAS_ENV:
    print("env_lseg.xlsx not found: RQ2 and Fig. 3 will be skipped (see data/env_lseg_template.xlsx).")

## 1. Data preparation

### 1.1 Loan weights

Each reported loan category is mapped to one or more ISIC Rev. 4 sections; a category spanning several sections is split equally. Household loans have no ENCORE activity, so they are left out of PBIS and reported separately. PBIS weights are shares of business loans.

In [ ]:
SECTIONS = list("ABCDEFGHIJKLMNOPQRS")
PARTS = SECTIONS + ["OTH"]
SEC_NAME = {"A": "Agriculture, forestry & fishing", "B": "Mining & quarrying", "C": "Manufacturing",
            "D": "Electricity & gas", "E": "Water & waste", "F": "Construction", "G": "Wholesale & retail trade",
            "H": "Transport & storage", "I": "Accommodation & food", "J": "Information & communication",
            "K": "Finance & insurance", "L": "Real estate", "M": "Professional services", "N": "Administrative services",
            "O": "Public administration", "P": "Education", "Q": "Health & social work", "R": "Arts & recreation",
            "S": "Other services", "OTH": "Others (unmapped)"}
GROUPS = {"Agriculture & mining": ["A", "B"], "Manufacturing": ["C"], "Utilities": ["D", "E"],
          "Construction & real estate": ["F", "L"], "Trade & hospitality": ["G", "I"], "Transport & ICT": ["H", "J"],
          "Finance & business services": ["K", "M", "N"], "Public & social services": ["O", "P", "Q", "R", "S"],
          "Others": ["OTH"]}

credit = pd.read_excel(PATH["credit"], sheet_name="credit")
credit = credit.rename(columns={"Reported sector": "Reported", "Mapped ISIC section": "Mapped",
                                "Share of gross loans": "Share"})[["Bank", "Country", "Year", "Reported", "Mapped", "Amount", "Share"]]
credit["Reported"] = credit["Reported"].str.strip()
def map_codes(text):
    t = str(text)
    if t.startswith("Household"): return ["HH"]
    if t.startswith("Others"): return ["OTH"]
    if t.startswith("Unallocated"): return ["GAP"]
    codes = re.findall(r"(?:^|\+ )([A-S]) –", t)
    assert codes, f"Unreadable mapping: {t}"
    return codes
credit["Codes"] = credit["Mapped"].map(map_codes)
BANKS = list(dict.fromkeys(credit["Bank"]))
COUNTRY = credit.drop_duplicates("Bank").set_index("Bank")["Country"]
check = credit.groupby(["Bank", "Year"])["Share"].sum()
assert np.allclose(check, 1, atol=1e-6), check[~np.isclose(check, 1)]

def loan_weights(id_others_household=ID_OTHERS_AS_HOUSEHOLD, sg_bc_real_estate=SG_BC_REAL_ESTATE, split="equal"):
    """Business-loan weights by ISIC section for every bank-year."""
    rows = []
    for (bank, ctry, year), g in credit.groupby(["Bank", "Country", "Year"], sort=False):
        acc = dict.fromkeys(PARTS + ["HH", "GAP"], 0.0)
        for _, r in g.iterrows():
            codes = list(r.Codes)
            if ctry == "Indonesia" and r.Reported == "Others":
                codes = ["HH"] if id_others_household else ["OTH"]
            if ctry == "Singapore" and r.Reported == "Building and construction":
                acc["F"] += r.Share * (1 - sg_bc_real_estate); acc["L"] += r.Share * sg_bc_real_estate
                continue
            if split == "first": codes = codes[:1]
            for c in codes: acc[c] += r.Share / len(codes)
        business = sum(acc[p] for p in PARTS)
        rows.append({"Bank": bank, "Country": ctry, "Year": year, "Household_share": acc["HH"],
                     "Business_share": business, "Gap_share": acc["GAP"], **{p: acc[p] / business for p in PARTS}})
    return pd.DataFrame(rows)

W = loan_weights()
KEYS = ["Bank", "Country", "Year"]
assert len(W) == 39 and np.allclose(W[PARTS].sum(1), 1)

# Table: mapping and loan composition
MAPPING = (credit.groupby(["Reported", "Mapped"])
                 .agg(Banks=("Bank", lambda s: ", ".join(dict.fromkeys(s))), Mean_share=("Share", "mean"))
                 .reset_index().sort_values("Mapped"))
save_tab(MAPPING, "A1_sector_mapping", index=False)
COMPOSITION = pd.DataFrame({
    "Country": COUNTRY,
    "Household % of gross loans": 100 * W.groupby("Bank").Household_share.mean(),
    "Others % of business loans": 100 * W.groupby("Bank").OTH.mean(),
    "Agri & mining % of business": 100 * W.groupby("Bank")[["A", "B"]].mean().sum(1),
    "Construction & real estate % of business": 100 * W.groupby("Bank")[["F", "L"]].mean().sum(1)}).loc[BANKS].round(1)
display(COMPOSITION); save_tab(COMPOSITION, "T1_loan_composition")
NUM["household_share_by_country"] = (100 * W.groupby("Country").Household_share.mean()).round(1).to_dict()
NUM["others_share_business_max"] = round(100 * W.OTH.max(), 1)

# 'Others' by bank-year
T_OTHERS = (100 * W.pivot(index="Bank", columns="Year", values="OTH")).loc[BANKS].round(1)
T_OTHERS.columns = [f"Others % of business loans {c}" for c in T_OTHERS.columns]
NUM["others_flagged_over_20pct"] = int((W.OTH > 0.20).sum())
print("Bank-years with Others > 20% of business loans:", NUM["others_flagged_over_20pct"])

### 1.2 ENCORE section scores

Ratings VL-VH are coded 1-5. N/A (pressure not relevant for the activity) is coded 0 and ND (no data) is treated as missing. An activity's score is the mean over its assessed pressures, and a section's score is the mean over its activities. S1-S8 are the alternative rules used in Section 2.4.

In [ ]:
enc = pd.read_excel(PATH["encore"], sheet_name="encore_raw", keep_default_na=False, na_values=[""])
raw = enc["rating_original"].astype(str).str.strip()
SECTION_OF = enc.drop_duplicates("activity").set_index("activity")["isic_section"]
NUM["encore"] = {"activities": int(enc.activity.nunique()), "pressures": int(enc.pressure.nunique()),
                 "ratings": int(len(enc)), "NA": int((raw == "N/A").sum()), "ND": int((raw == "ND").sum())}

def rated(na="zero", nd="zero"):
    r = enc["rating"].map(RATING).astype(float)
    r[raw.eq("N/A")] = 0.0 if na == "zero" else np.nan
    r[raw.eq("ND")] = 0.0 if nd == "zero" else np.nan
    r[r.isna() & ~raw.isin(["N/A", "ND"])] = 0.0
    return enc.assign(r=r)

def section_scores(activity_score, agg="mean", others="mean"):
    df = pd.DataFrame({"s": activity_score, "sec": SECTION_OF.reindex(activity_score.index)}).dropna()
    s = df.groupby("sec")["s"].agg(agg).reindex(SECTIONS)
    s["OTH"] = {"mean": s.mean(), "min": s.min(), "median": s.median(), "max": s.max()}[others]
    return s[PARTS]

act_mean = lambda E: E.groupby("activity")["r"].mean()
E0 = rated(na="zero", nd="excl")                       # BASELINE: N/A = 0, ND = missing
share_high = (enc.assign(h=enc["rating"].isin(["H", "VH"]).astype(float).where(~raw.eq("ND")))
                 .groupby("activity")["h"].mean())     # share among assessed pressures
SCORES = pd.DataFrame({
    "S0": section_scores(act_mean(E0)),
    "S1": section_scores(act_mean(E0), others="min"),
    "S2": section_scores(act_mean(E0), others="median"),
    "S3": section_scores(act_mean(E0), others="max"),
    "S4": section_scores(E0.groupby("activity")["r"].max()),
    "S5": section_scores(act_mean(E0), agg="median"),
    "S6": section_scores(act_mean(rated(na="zero", nd="zero"))),
    "S7": section_scores(act_mean(rated(na="excl", nd="excl"))),
    "S8": section_scores(share_high)})
SCENARIO = {"S0": "Baseline (N/A = 0, ND = missing)", "S1": "Others = lowest section score",
            "S2": "Others = median section score", "S3": "Others = highest section score",
            "S4": "Activity = highest pressure rating", "S5": "Section = median of activities",
            "S6": "ND = 0", "S7": "N/A and ND excluded", "S8": "Share of assessed pressures rated H/VH"}
NUM["encore"]["activities_with_ND"] = int(enc.loc[raw.eq("ND"), "activity"].nunique())
NUM["encore"]["activities_all_ND"] = int((act_mean(E0).isna()).sum())
T_SCORES = SCORES.copy(); T_SCORES.insert(0, "Activities", SECTION_OF.value_counts().reindex(PARTS))
T_SCORES.insert(0, "Section", [SEC_NAME[p] for p in PARTS])
display(T_SCORES.round(3)); save_tab(T_SCORES, "T2_encore_section_scores")

## 2. RQ1: PBIS, drivers and robustness

PBIS for a bank-year is the sum of section scores weighted by the bank's business-loan shares.

In [ ]:
def pbis(weights, scores): return (weights[PARTS].values * scores[PARTS].values[None, :]).sum(1)
PB = W[KEYS].copy()
for s in SCORES: PB[s] = pbis(W, SCORES[s])
CONTRIB = pd.DataFrame(W[PARTS].values * SCORES["S0"].values[None, :], columns=PARTS)

BANK = PB.groupby("Bank")[list(SCORES)].mean().loc[BANKS]          # bank means 2023–2025
BANK["Country"] = COUNTRY
T_PBIS = PB.pivot(index="Bank", columns="Year", values="S0").loc[BANKS]
T_PBIS["Mean"] = BANK.S0; T_PBIS["Rank"] = BANK.S0.rank(ascending=False).astype(int); T_PBIS.insert(0, "Country", COUNTRY)
display(T_PBIS.round(3)); save_tab(T_PBIS, "T3_pbis_by_bank")
NUM["pbis"] = {"bank_mean": BANK.S0.round(3).to_dict(), "country_mean": PB.groupby("Country").S0.mean().round(3).to_dict(),
               "overall_mean": round(PB.S0.mean(), 3), "range": [round(BANK.S0.min(), 3), round(BANK.S0.max(), 3)],
               "within_bank_sd": round(PB.groupby("Bank").S0.std().mean(), 3), "between_bank_sd": round(BANK.S0.std(), 3)}

### 2.1 Contributions and drivers

In [ ]:
CB = pd.concat([W[["Bank"]], CONTRIB], axis=1).groupby("Bank")[PARTS].mean().loc[BANKS]   # bank-mean contributions
CG = pd.DataFrame({g: CB[v].sum(1) for g, v in GROUPS.items()})                         # by sector group
WB = W.groupby("Bank")[PARTS].mean().loc[BANKS]

# Between-bank variance decomposition
y = CB.sum(1)
VARSHARE = CG.apply(lambda c: np.cov(c, y)[0, 1]) / y.var()
def r2(y, X):
    X = np.column_stack([np.ones(len(y)), X]); b = np.linalg.lstsq(X, y, rcond=None)[0]
    return 1 - ((y - X @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum()
R2 = {"Agriculture & mining share": r2(y.values, WB[["A", "B"]].sum(1).values[:, None]),
      "Real-estate share": r2(y.values, WB[["L"]].values),
      "Household share of gross loans": r2(y.values, W.groupby("Bank").Household_share.mean().loc[BANKS].values[:, None]),
      "Real-estate + household shares": r2(y.values, np.column_stack([WB["L"].values,
                                                     W.groupby("Bank").Household_share.mean().loc[BANKS].values])),
      "Country": r2(y.values, pd.get_dummies(COUNTRY.loc[BANKS], drop_first=True).astype(float).values)}
# 'Others' contribution to PBIS
oth_c = (CONTRIB["OTH"] / PB["S0"]).groupby(W.Bank).mean().loc[BANKS]
T_OTHERS["Share of PBIS from Others % (mean)"] = (100 * oth_c).round(1)
display(T_OTHERS); save_tab(T_OTHERS, "T1b_others_by_bank_year")
NUM["others_contribution_max_pct"] = round(100 * oth_c.max(), 1)
T_DRIVERS = pd.DataFrame({"Mean contribution": CG.mean(), "Share of between-bank variance": VARSHARE}).sort_values(
    "Share of between-bank variance", ascending=False)
display(T_DRIVERS.round(3)); print({k: round(v, 3) for k, v in R2.items()})
save_tab(T_DRIVERS, "T4_drivers")
NUM["drivers"] = {"R2": {k: round(v, 3) for k, v in R2.items()}, "variance_share": VARSHARE.round(3).to_dict()}

# Country profile
cc = pd.concat([W[["Country"]], CONTRIB], axis=1).groupby("Country")[PARTS].sum()
T_COUNTRY = pd.DataFrame({g: cc[v].sum(1) for g, v in GROUPS.items()})
T_COUNTRY = (100 * T_COUNTRY.div(T_COUNTRY.sum(1), axis=0)).round(1)
T_COUNTRY.insert(0, "Household % of gross loans", (100 * W.groupby("Country").Household_share.mean()).round(1))
T_COUNTRY.insert(0, "Mean PBIS", PB.groupby("Country").S0.mean().round(3))
display(T_COUNTRY.T); save_tab(T_COUNTRY.T, "T5_country_profile")

### 2.2 Change 2023-2025

In [ ]:
rows = []
for b in BANKS:
    g = PB[PB.Bank == b].sort_values("Year"); w = W[W.Bank == b].sort_values("Year")[PARTS]
    d = (w.iloc[-1] - w.iloc[0]) * SCORES.S0
    dg = pd.Series({k: d[v].sum() for k, v in GROUPS.items()})
    rows.append({"Bank": b, "Country": COUNTRY[b], "PBIS 2023": g.S0.iloc[0], "PBIS 2025": g.S0.iloc[-1],
                 "Change": g.S0.iloc[-1] - g.S0.iloc[0], "Main driver": f"{dg.abs().idxmax()} ({dg[dg.abs().idxmax()]:+.3f})"})
T_TREND = pd.DataFrame(rows).set_index("Bank").sort_values("Change")
display(T_TREND.round(3)); save_tab(T_TREND, "T6_change_2023_2025")
NUM["trend"] = T_TREND[["Change", "Main driver"]].round(3).to_dict("index")

### 2.3 Ranking agreement (Kendall τ with exact pair counts; 13 banks = 78 pairs)

In [ ]:
def compare_rankings(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); c = d = t = 0
    for i, j in itertools.combinations(range(len(a)), 2):
        s = np.sign(a[i] - a[j]) * np.sign(b[i] - b[j]); c += s > 0; d += s < 0; t += s == 0
    shift = np.abs(stats.rankdata(-a) - stats.rankdata(-b))
    return {"tau": stats.kendalltau(a, b)[0], "discordant_pairs": int(d), "tied_pairs": int(t), "max_rank_shift": int(shift.max())}

### 2.4 Robustness to ENCORE scoring rules

In [ ]:
T_SCORING = pd.DataFrame([{"Scenario": s, "Rule": SCENARIO[s], **compare_rankings(BANK.S0, BANK[s])}
                          for s in list(SCORES)[1:]]).set_index("Scenario")
display(T_SCORING.round(3)); save_tab(T_SCORING, "T7_robustness_scoring")
NUM["robust_scoring"] = {"tau_min": round(T_SCORING.tau.min(), 3), "tau_max": round(T_SCORING.tau.max(), 3),
                         "table": T_SCORING.round(3).to_dict("index")}

### 2.5 Robustness to sector-mapping assumptions

In [ ]:
MAP_SCEN = [("Baseline", {}),
            ("Indonesian 'Others' = unmapped business loans", dict(id_others_household=False)),
            ("SG building & construction = 100% construction", dict(sg_bc_real_estate=0.0)),
            ("SG building & construction = 100% real estate", dict(sg_bc_real_estate=1.0)),
            ("Combined categories = first section only", dict(split="first")),
            ("Indonesian 'Others' unmapped + SG 100% real estate", dict(id_others_household=False, sg_bc_real_estate=1.0))]
MAP_PBIS, rows = {}, []
for name, kw in MAP_SCEN:
    w = loan_weights(**kw); p = pd.Series(pbis(w, SCORES.S0)).groupby(w.Bank.values).mean().loc[BANKS]
    MAP_PBIS[name] = p; cm = p.groupby(COUNTRY.loc[BANKS]).mean()
    rows.append({"Scenario": name, **compare_rankings(BANK.S0, p),
                 "PBIS Indonesia": cm["Indonesia"], "PBIS Malaysia": cm["Malaysia"], "PBIS Singapore": cm["Singapore"]})
T_MAPPING = pd.DataFrame(rows).set_index("Scenario")
display(T_MAPPING.round(3)); save_tab(T_MAPPING, "T8_robustness_mapping")
NUM["robust_mapping"] = T_MAPPING.round(3).to_dict("index")

### 2.6 External benchmark: share of business lending to high-materiality activities

In [ ]:
HIGH = section_scores(enc.assign(h=enc["rating"].isin(["H", "VH"])).groupby("activity")["h"].max().astype(float))
PB["HighMat"] = pbis(W, HIGH)
BENCH = PB.groupby("Bank")[["S0", "HighMat"]].mean().loc[BANKS]
NUM["benchmark"] = {"spearman": round(stats.spearmanr(BENCH.S0, BENCH.HighMat)[0], 3),
                    **{k: (round(v, 3) if isinstance(v, float) else v) for k, v in compare_rankings(BENCH.S0, BENCH.HighMat).items()},
                    "high_materiality_share_by_bank": (100 * BENCH.HighMat).round(1).to_dict()}
print(NUM["benchmark"])

## 3. RQ2: PBIS and the LSEG Environmental Pillar Score

The main test uses the 13 bank means. p-values come from permutations and the confidence interval from a bootstrap over banks. For the pooled 39 bank-years, each bank's three years are permuted as one block. The within-country test shuffles scores only inside each country.

In [ ]:
if HAS_ENV:
    env = pd.read_excel(PATH["env"])
    env = env.rename(columns={env.columns[0]: "Bank", env.columns[1]: "Country"})
    env["Bank"] = env["Bank"].replace({"Bank Mandiri": "Mandiri", "Hong Leong": "Hong Leong Bank"}).str.strip()
    ENV = env.melt(id_vars=["Bank", "Country"], var_name="Year", value_name="ENV").drop(columns="Country")
    ENV["Year"] = ENV["Year"].astype(int)
    assert set(ENV.Bank) == set(BANKS), set(ENV.Bank) ^ set(BANKS)
    D = PB.merge(ENV, on=["Bank", "Year"])
    M = D.groupby("Bank")[["S0", "ENV"]].mean().loc[BANKS].rename(columns={"S0": "PBIS"}); M["Country"] = COUNTRY
    T_ENV = env.set_index("Bank").loc[BANKS]
    display(T_ENV); save_tab(T_ENV, "T9_lseg_environmental_score")

    def corr_rows(a, B):
        a = a - a.mean(); B = B - B.mean(1, keepdims=True)
        return (B @ a) / (np.sqrt((B ** 2).sum(1)) * np.sqrt((a ** 2).sum()) + 1e-300)
    def p_two_sided(obs, null): return (1 + np.sum(np.abs(null) >= abs(obs) - 1e-12)) / (len(null) + 1)

    def bank_level_test(x, z, P=N_PERM, B=N_BOOT):
        x, z = np.asarray(x), np.asarray(z); rho = stats.spearmanr(x, z)[0]
        null = corr_rows(stats.rankdata(x), np.array([stats.rankdata(rng.permutation(z)) for _ in range(P)]))
        boot = []
        for _ in range(B):
            i = rng.integers(0, len(x), len(x))
            if np.ptp(x[i]) > 0 and np.ptp(z[i]) > 0: boot.append(stats.spearmanr(x[i], z[i])[0])
        return {"n": len(x), "rho": rho, "p_perm": p_two_sided(rho, null),
                "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5)}

    RQ2 = {"bank_means": bank_level_test(M.PBIS, M.ENV)}
    # pooled bank-years, cluster-aware
    P_ = np.vstack([D[D.Bank == b].sort_values("Year").S0.values for b in BANKS])
    E_ = np.vstack([D[D.Bank == b].sort_values("Year").ENV.values for b in BANKS])
    rho_pool = stats.spearmanr(P_.ravel(), E_.ravel())[0]
    null = corr_rows(stats.rankdata(P_.ravel()), np.array([stats.rankdata(E_[rng.permutation(13)].ravel()) for _ in range(N_PERM)]))
    boot = [stats.spearmanr(P_[i].ravel(), E_[i].ravel())[0] for i in (rng.integers(0, 13, 13) for _ in range(N_BOOT))]
    RQ2["pooled_bank_years"] = {"n": 39, "rho": rho_pool, "p_naive": stats.spearmanr(P_.ravel(), E_.ravel())[1],
                                "p_perm_by_bank": p_two_sided(rho_pool, null),
                                "ci_low": np.nanpercentile(boot, 2.5), "ci_high": np.nanpercentile(boot, 97.5)}
    RQ2["by_year"] = {int(yr): {"rho": stats.spearmanr(g.S0, g.ENV)[0], "p": stats.spearmanr(g.S0, g.ENV)[1]} for yr, g in D.groupby("Year")}
    # within country
    RQ2["within_country"] = {c: {"n": len(g), "rho": stats.spearmanr(g.PBIS, g.ENV)[0]} for c, g in M.groupby("Country")}
    groups = [np.where(M.Country.values == c)[0] for c in M.Country.unique()]
    def centred_ranks(v):
        out = np.empty(len(v))
        for ix in groups: r = stats.rankdata(v[ix]); out[ix] = r - r.mean()
        return out
    cp = centred_ranks(M.PBIS.values); r_within = np.corrcoef(cp, centred_ranks(M.ENV.values))[0, 1]
    perm = np.empty((N_PERM, 13))
    for t in range(N_PERM):
        v = M.ENV.values.copy()
        for ix in groups: v[ix] = v[rng.permutation(ix)]
        perm[t] = centred_ranks(v)
    RQ2["within_country_pooled"] = {"r": r_within, "p_stratified_perm": p_two_sided(r_within, corr_rows(cp, perm))}
    RQ2["country_means"] = M.groupby("Country")[["PBIS", "ENV"]].mean().round(2).to_dict("index")
    # power
    se = np.sqrt(1.06 / (13 - 3)); zc = stats.norm.ppf(.975)
    power = lambda r: 1 - stats.norm.cdf(zc - np.arctanh(abs(r)) / se) + stats.norm.cdf(-zc - np.arctanh(abs(r)) / se)
    RQ2["power"] = {"critical_rho_alpha05": np.tanh(zc * se), "power_at_observed": power(RQ2["bank_means"]["rho"]),
                    "min_detectable_rho_80": brentq(lambda r: power(r) - .8, .05, .99)}
    # robustness to scoring rules
    RQ2["by_scoring_rule"] = {s: bank_level_test(BANK[s], M.ENV, P=5000, B=500) for s in SCORES}
    RQ2["by_mapping"] = {k: bank_level_test(v, M.ENV, P=5000, B=500) for k, v in MAP_PBIS.items()}
    T_RQ2 = pd.DataFrame({**{f"Scoring {s}": RQ2["by_scoring_rule"][s] for s in SCORES},
                          **{f"Mapping: {k}": v for k, v in RQ2["by_mapping"].items()}}).T[["rho", "p_perm", "ci_low", "ci_high"]]
    print(json.dumps({k: RQ2[k] for k in ["bank_means", "pooled_bank_years", "by_year", "within_country",
                                           "within_country_pooled", "country_means", "power"]}, indent=1, default=float))
    display(T_RQ2.round(3)); save_tab(T_RQ2, "T10_rq2_robustness")
    NUM["rq2"] = json.loads(json.dumps({k: v for k, v in RQ2.items() if not k.startswith("by_") or k == "by_year"}, default=float))
    NUM["rq2"]["rho_range_scoring_rules"] = [round(T_RQ2.iloc[:9].rho.min(), 3), round(T_RQ2.iloc[:9].rho.max(), 3)]
    NUM["rq2"]["rho_range_mapping"] = [round(T_RQ2.iloc[9:].rho.min(), 3), round(T_RQ2.iloc[9:].rho.max(), 3)]
else:
    print("Skipped: env_lseg.xlsx not available.")

### 3.1 Rank gap

Gap = scaled PBIS rank + scaled environmental-score rank - 1. Positive and negative gaps are both read as misalignment, and each gap is compared with random pairings rather than a fixed cut-off.

In [ ]:
if HAS_ENV:
    def rank_gap(pbis_values, env_values):
        n = len(pbis_values)
        return (stats.rankdata(pbis_values) - 1) / (n - 1) + (stats.rankdata(env_values) - 1) / (n - 1) - 1
    GAP = pd.DataFrame({"Country": COUNTRY, "PBIS": M.PBIS, "ENV": M.ENV, "Gap": rank_gap(M.PBIS, M.ENV)}, index=BANKS)
    rp = (stats.rankdata(M.PBIS) - 1) / 12; re_ = (stats.rankdata(M.ENV) - 1) / 12
    null = rp[None, :] + np.array([rng.permutation(re_) for _ in range(N_PERM)]) - 1
    GAP["p_random"] = (1 + (np.abs(null) >= np.abs(GAP.Gap.values)[None, :] - 1e-12).sum(0)) / (N_PERM + 1)
    allg = pd.DataFrame({s: rank_gap(BANK[s], M.ENV) for s in SCORES}, index=BANKS)   # all scoring rules S0-S8
    GAP["Gap_min"], GAP["Gap_max"] = allg.min(1), allg.max(1)
    GAP = GAP.reindex(GAP.Gap.abs().sort_values(ascending=False).index)
    display(GAP.round(3)); save_tab(GAP, "T11_rank_gap")
    NUM["gap"] = GAP.round(3).head(4).to_dict("index")
else:
    print("Skipped: env_lseg.xlsx not available.")

## 4. RQ3: Bank-level clustering

Clustering is done on the 13 bank means, not on bank-years. Loan shares are grouped into nine sector groups, square-root (Hellinger) transformed and reduced with PCA before Ward clustering. Stability is checked by resampling banks. The bank-year version is computed only for comparison.

In [ ]:
WG = pd.DataFrame({g: W[v].sum(1) for g, v in GROUPS.items()}); WG["Bank"] = W.Bank
FEAT = WG.groupby("Bank").mean().loc[BANKS]
def pca_fit(X):
    full = PCA().fit(X)
    k = max(2, int(np.searchsorted(np.cumsum(full.explained_variance_ratio_), PCA_VAR - 1e-12) + 1))
    p = PCA(k).fit(X); return p, p.transform(X)
ward = lambda Z, k: AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Z)
X = np.sqrt(FEAT.values); PCA_B, Z = pca_fit(X)
rows, LABELS = [], {}
for k in K_RANGE:
    lab = ward(Z, k); LABELS[k] = lab; aris = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, 13, 13)
        if len(np.unique(idx)) <= k: continue
        _, Zb = pca_fit(X[idx]); lb = ward(Zb, k); _, first = np.unique(idx, return_index=True)
        aris.append(adjusted_rand_score(lab[idx[first]], lb[first]))
    rows.append({"k": k, "Silhouette": silhouette_score(Z, lab), "Cluster sizes": str(sorted(np.bincount(lab).tolist(), reverse=True)),
                 "Bootstrap ARI (mean)": np.mean(aris), "Bootstrap ARI (2.5%)": np.percentile(aris, 2.5),
                 "ARI with country": adjusted_rand_score(COUNTRY.loc[BANKS], lab),
                 "ARI Ward vs K-Means": adjusted_rand_score(KMeans(k, n_init=50, random_state=SEED).fit_predict(Z), lab)})
T_CLUST = pd.DataFrame(rows).set_index("k")
# For comparison only: the same pipeline on the 39 bank-years, with a bootstrap that resamples
# bank-years. Near-identical yearly copies of each bank inflate both numbers.
Xp = np.sqrt(WG[list(GROUPS)].values); _, Zp = pca_fit(Xp)
for k in K_RANGE:
    lab_p = ward(Zp, k); aris = []
    for _ in range(500):
        idx = rng.integers(0, 39, 39); _, Zb = pca_fit(Xp[idx]); lb = ward(Zb, k); _, first = np.unique(idx, return_index=True)
        aris.append(adjusted_rand_score(lab_p[idx[first]], lb[first]))
    T_CLUST.loc[k, "Silhouette (39 bank-years)"] = silhouette_score(Zp, lab_p)
    T_CLUST.loc[k, "Bootstrap ARI, bank-years resampled (mean)"] = np.mean(aris)
K_BEST = int(T_CLUST.Silhouette.idxmax())
LOAD = pd.DataFrame(PCA_B.components_.T, index=FEAT.columns, columns=[f"PC{i+1}" for i in range(PCA_B.n_components_)])
MEMBER = pd.Series(LABELS[K_BEST], index=BANKS)
order = MEMBER.groupby(MEMBER).apply(lambda s: BANK.S0[s.index].mean()).sort_values(ascending=False).index
MEMBER = MEMBER.map({old: new + 1 for new, old in enumerate(order)})
PROFILE = (100 * FEAT).groupby(MEMBER).mean().round(1)
PROFILE.insert(0, "Mean PBIS", BANK.S0.groupby(MEMBER).mean().round(3))
PROFILE.insert(0, "Banks", MEMBER.groupby(MEMBER).apply(lambda s: ", ".join(s.index)))
display(T_CLUST.round(3)); display(LOAD.round(3)); display(PROFILE.T)
save_tab(T_CLUST, "T12_clustering"); save_tab(LOAD, "T12b_pca_loadings"); save_tab(PROFILE.T, "T13_cluster_profiles")
NUM["rq3"] = {"k_best": K_BEST, "n_pcs": int(PCA_B.n_components_), "pca_variance": round(PCA_B.explained_variance_ratio_.sum(), 3),
              "table": T_CLUST.round(3).to_dict("index"), "membership": MEMBER.to_dict(),
              "second_best_k": int(T_CLUST.Silhouette.drop(K_BEST).idxmax())}
for k in K_RANGE:
    print(f"k={k}:", " | ".join(", ".join(np.array(BANKS)[LABELS[k] == c]) for c in np.unique(LABELS[k])))

## 5. Figures

In [ ]:
# Fig. 2 — (a) sector-group contributions, (b) PBIS vs agriculture & mining share
order = BANK.S0.sort_values(ascending=False).index
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.5), gridspec_kw={"width_ratios": [1.55, 1]})
CG.loc[order].plot.bar(stacked=True, ax=ax[0], color=plt.get_cmap("tab10")(np.arange(9)), width=.8, edgecolor="k", linewidth=.2)
ax[0].set_title("(a) Sector-group contributions to PBIS (2023–2025 mean)"); ax[0].set_xlabel(""); ax[0].set_ylabel("PBIS")
ax[0].set_xticklabels(order, rotation=45, ha="right")
ax[0].legend(fontsize=5.5, ncol=3, loc="upper center", bbox_to_anchor=(0.72, -0.48), frameon=False)
xs = 100 * WB[["A", "B"]].sum(1)
for b in BANKS:
    ax[1].scatter(xs[b], BANK.S0[b], c=COLORS[COUNTRY[b]], s=24, edgecolor="k", lw=.4)
    ax[1].annotate(b, (xs[b], BANK.S0[b]), xytext=(3, 2), textcoords="offset points", fontsize=5.8)
for c, col in COLORS.items(): ax[1].scatter([], [], c=col, label=c, edgecolor="k", lw=.4)
ax[1].legend(loc="upper left"); ax[1].set_xlabel("Agriculture & mining share of business loans (%)"); ax[1].set_ylabel("PBIS")
ax[1].set_title(f"(b) PBIS vs agriculture & mining share ($R^2$ = {R2['Agriculture & mining share']:.2f})")
plt.tight_layout(); save_fig("Fig2_pbis_contributions")

if HAS_ENV:
    # Fig. 3 — (a) PBIS vs LSEG Environmental Pillar Score, (b) rank gap
    r = RQ2["bank_means"]
    fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
    for b in BANKS:
        ax[0].scatter(M.ENV[b], M.PBIS[b], c=COLORS[COUNTRY[b]], s=24, edgecolor="k", lw=.4)
        ax[0].annotate(b, (M.ENV[b], M.PBIS[b]), xytext=(3, 2), textcoords="offset points", fontsize=5.8)
    for c, col in COLORS.items(): ax[0].scatter([], [], c=col, label=c, edgecolor="k", lw=.4)
    ax[0].legend(loc="lower left"); ax[0].set_xlabel("LSEG Environmental Pillar Score (2023–2025 mean)"); ax[0].set_ylabel("PBIS")
    ax[0].set_title(f"(a) Spearman ρ = {r['rho']:.2f}, p = {r['p_perm']:.3f}, 95% CI [{r['ci_low']:.2f}, {r['ci_high']:.2f}]")
    g = GAP.sort_values("Gap"); yy = np.arange(len(g))
    ax[1].barh(yy, g.Gap, color=[COLORS[COUNTRY[b]] for b in g.index], alpha=.75, edgecolor="k", lw=.3)
    ax[1].errorbar(g.Gap, yy, xerr=[g.Gap - g.Gap_min, g.Gap_max - g.Gap], fmt="none", ecolor="k", lw=.7, capsize=1.5)
    ax[1].axvline(0, color="k", lw=.6); ax[1].set_yticks(yy); ax[1].set_yticklabels(g.index)
    ax[1].set_xlabel("Rank gap (bars: baseline; whiskers: S0–S8)"); ax[1].set_title("(b) PBIS–environmental score rank gap")
    plt.tight_layout(); save_fig("Fig3_pbis_lseg")

# Fig. 4 — (a) dendrogram, (b) PCA biplot
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
dendrogram(linkage(Z, "ward"), labels=BANKS, ax=ax[0], leaf_font_size=6.3, leaf_rotation=60, color_threshold=0)
ax[0].set_title("(a) Ward dendrogram of 13 banks"); ax[0].set_ylabel("Merge distance")
for b, z in zip(BANKS, Z):
    ax[1].scatter(z[0], z[1], c=COLORS[COUNTRY[b]], s=24, edgecolor="k", lw=.4)
    ax[1].annotate(b, z[:2], xytext=(3, 2), textcoords="offset points", fontsize=5.8)
top = LOAD[["PC1", "PC2"]].abs().sum(1).nlargest(5).index
sc = .8 * min(np.abs(Z[:, 0]).max() / LOAD.loc[top, "PC1"].abs().max(), np.abs(Z[:, 1]).max() / LOAD.loc[top, "PC2"].abs().max())
for s in top:
    ax[1].annotate("", xy=(LOAD.loc[s, "PC1"] * sc, LOAD.loc[s, "PC2"] * sc), xytext=(0, 0),
                   arrowprops=dict(arrowstyle="->", color="grey", lw=.8))
    ax[1].text(LOAD.loc[s, "PC1"] * sc * 1.1, LOAD.loc[s, "PC2"] * sc * 1.1, s, fontsize=5.5, color="dimgrey", ha="center")
ev = PCA_B.explained_variance_ratio_
ax[1].set_xlabel(f"PC1 ({100 * ev[0]:.0f}%)"); ax[1].set_ylabel(f"PC2 ({100 * ev[1]:.0f}%)")
ax[1].set_title("(b) PCA of lending composition (Hellinger)")
for c, col in COLORS.items(): ax[1].scatter([], [], c=col, label=c, edgecolor="k", lw=.4)
ax[1].legend(loc="best")
plt.tight_layout(); save_fig("Fig4_clustering")
print("Figures saved to", FIG)

## 6. Export

In [ ]:
def to_py(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return round(float(o), 4)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
NUM["settings"] = {"encore": ENCORE_VERSION, "seed": SEED, "n_boot": N_BOOT, "n_perm": N_PERM,
                   "indonesian_others_as_household": ID_OTHERS_AS_HOUSEHOLD, "sg_building_construction_real_estate_share": SG_BC_REAL_ESTATE}
json.dump(NUM, open(f"{OUT}/paper_numbers.json", "w"), indent=1, default=to_py)
with pd.ExcelWriter(f"{OUT}/PBIS_results.xlsx") as xw:
    for f in sorted(glob.glob(f"{TAB}/*.csv")):
        pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[:-4][:31], index=False)
print("All outputs in", OUT)
if os.path.isdir("/content"):
    import shutil; shutil.make_archive("/content/PBIS_outputs", "zip", OUT); print("Download /content/PBIS_outputs.zip")